# 02 · Calibración de N, U y umbral de fraude (§8.3)

Valores actuales en `services/src/sofia_services/settings.py` (SIM): `DISPUTE_WINDOW_DAYS=90`,
`AMOUNT_THRESHOLD_USD=500`, `FRAUD_SCORE_THRESHOLD=0.8`. Este notebook los justifica o propone cambios (PR a SIM).

- **N (ventana):** días entre transacción y queja → percentil que cubre la mayoría de disputas legítimas.
- **U (monto):** `claimed_amount` en USD → qué % de disputas va a auto vs humano.
- **Fraude:** `fraud_score` contra `is_fraud` → precisión/recall del umbral.

Salida: `analysis/results/policy_calibration.json`.

In [ ]:
from common import connect, save_result, show

con = connect()

## U: monto reclamado en USD

In [ ]:
# TODO DS: confirmar columnas y dirección de daily_exchange_rates (DESCRIBE daily_exchange_rates).
amounts = show(con, "claimed_amount por moneda (disputas)", """
    SELECT currency, count(*) AS n,
           quantile_cont(claimed_amount, [0.5, 0.75, 0.8, 0.9, 0.95, 0.99]) AS p50_75_80_90_95_99
    FROM complaints
    WHERE subcategory IN ('Cargo no reconocido', 'Cobro indebido') AND claimed_amount IS NOT NULL
    GROUP BY 1 ORDER BY n DESC""")

## N: días entre transacción y queja
Usa el vínculo queja → transacción del notebook 01 §4.

In [ ]:
# TODO DS: distribución de (creation_date - transaction_date) en disputas ligadas; percentil 95.

## Umbral de fraude

In [ ]:
show(con, "fraud_score vs is_fraud", """
    SELECT round(fraud_score, 1) AS bucket, count(*) AS n, round(avg(is_fraud::INT), 3) AS p_fraude
    FROM transactions GROUP BY 1 ORDER BY 1""")

## Decisión

In [ ]:
save_result("policy_calibration", {
    "N_days": {"value": None, "current": 90, "rationale": "TODO"},
    "U_usd": {"value": None, "current": 500, "rationale": "TODO"},
    "fraud_score_threshold": {"value": None, "current": 0.8, "rationale": "TODO"},
})